### Golden Search Section

It's the easiest algorithm ever to find the maximum (or the minimum) of a function $f(x)$ with $x \in [a, b]$.

Note: the function must be unimodal when $x \in [a, b]$

In [1]:
import numpy as np
from scipy.optimize import fminbound

from scipy.constants import atomic_mass as u # atomic mass unit
from scipy.constants import golden_ratio as phi # phi constant
from scipy.constants import k # Boltzmann constant

In [2]:
def GSS(f, a0, b0, eps = 1e-06, *args): # Implements the GSS algorithm with upper bound absolute error

    a, b = a0, b0 # Initial guess

    # Two interior points (mirror-symmetric about the midpoint)
    x1 = b - (b - a) / phi
    x2 = a + (b - a) / phi
    f1, f2 = f(x1, *args), f(x2, *args)

    j = 0 # Number of iterations

    while (b - a) / 2.0 >= eps: # Shrink the interval until the half-width drops below eps
        j += 1

        if f1 < f2: # Max lies in [x1, b]
            a = x1
            x1, f1 = x2, f2 # Reuse the surviving point
            x2 = a + (b - a) / phi
            f2 = f(x2, *args)
        else: # Max lies in [a, x2]
            b = x2
            x2, f2 = x1, f1 # Reuse the surviving point
            x1 = b - (b - a) / phi
            f1 = f(x1, *args)

    x = (a + b) / 2.0 # Best estimate: midpoint
    err = (b - a) / 2.0 # Error: |x - x*| < (b - a) / 2

    print(f"Number of iterations = {j}")
    return x, err

Let's test it. Compute the most likely velocity of the Maxwell-Boltzmann distribution for Oxygen at room temperature

In [ ]:
boltzmann = lambda v, m, T: 4.0 * np.pi * (np.sqrt(m / (2.0 * np.pi * k * T))) ** 3 * v ** 2 * np.exp(- m * v ** 2 / (2.0 * k * T)) # Maxwell-Boltzmann distribution

m = 16.0 * u # Oxygen mass
T = 20.0 + 273.15 # 20 °C

a0 = 400.0 # 400 m/s
b0 = 600.0 # 600 m/s

GSS(boltzmann, a0, b0, 1e-02, m, T) # Finds the maximum velocity. Low precision

Number of iterations = 20


(551.969256989563, 0.006610696135226135)

Result:

$$
    v_{max} = 551,969 \pm 0,007 \, \text{m/s}
$$

Let's see how much accuracy we can reach

In [ ]:
res = GSS(boltzmann, a0, b0, 1e-12, m, T) # High precision
print(f"res = {res[0]:.12f} +- {res[1]:.12f}")

Number of iterations = 67
res = 551.972000579784 +- 0.000000000001


In [5]:
res = fminbound( # From Scipy
    func = lambda v, m, T: - boltzmann(v, m, T),
    x1 = a0,
    x2 = b0,
    args = (m, T),
    xtol = 1e-12
)

print(f"res = {res:.12f}")

res = 551.972000788060
